# 02 · 실험 관리(MLflow) + 모델 성능 비교

MLOps 파이프라인 단계. **요구사항 #5(실험 관리/AutoML 대안) + #6(모델 평가·성능 비교)** 담당.

- **무엇을 비교하나:** 생성 모델 `gpt-4o-mini` vs `gpt-4o` × 코칭 프롬프트 `v1/v2/v3`를
  동일 골든셋(30 케이스)에 대해 돌려 **Question Depth / 환각률 / 비용 / 지연**을 비교한다.
- **AutoML 대신 '실험 관리':** 본 과제는 모델 학습이 없으므로(No-Training),
  하이퍼파라미터 탐색 대신 **모델 × 프롬프트 조합을 체계적으로 탐색**하고 MLflow로 추적한다.
  → 이것이 LLM 서비스에서의 '실험 관리'다.
- **추적 도구:** MLflow(로컬 파일 스토어 `./mlruns`). 미설치 시 `experiments/runs.csv`로 폴백.

> 라이브 모드는 `OPENAI_API_KEY` 필요. 없으면 모델별 품질 프로파일을 반영한 **오프라인 모의**로
> 전체 실험·추적·비교 파이프라인을 재현한다.

In [ ]:
import sys, os, json, random, time
sys.path.append('..')
import pandas as pd
from src import data_prep as dp

MODELS = ['gpt-4o-mini', 'gpt-4o']     # 비교 대상 생성 모델
STYLES = ['v1', 'v2', 'v3']
JUDGE_MODEL = 'gpt-4o'                  # 판정 모델(생성과 분리)
W = {'openness':0.2,'groundedness':0.3,'specificity':0.2,'provocativeness':0.3}

USE_API = bool(os.getenv('OPENAI_API_KEY'))
random.seed(11)

# 골든셋(외부화된 버전 관리 데이터) 로드
gs = json.load(open('../data/golden_set.json', encoding='utf-8'))
GOLDEN = gs['cases']
print(f'golden: {len(GOLDEN)} | models: {MODELS} | styles: {STYLES}')
print('MODE =', 'LIVE API' if USE_API else 'OFFLINE (mock)')

## 1. 생성 + 채점 (모델 × 스타일)

각 (모델, 스타일, 케이스)에 대해 코치 응답을 생성하고 4축으로 채점한다.
오프라인 모의는 모델별 품질·비용·지연 프로파일을 반영한다(gpt-4o가 품질↑ 비용↑ 지연↑).

In [ ]:
MODEL_PROFILE = {  # 오프라인 모의용 (상대적 경향만 표현)
    'gpt-4o-mini': {'depth_bonus': 0.0, 'cost': 0.0011, 'lat': 2000, 'fab_base': 0.30},
    'gpt-4o':      {'depth_bonus': 0.4, 'cost': 0.0090, 'lat': 2800, 'fab_base': 0.10},
}
STYLE_BASE = {'v1': 3.0, 'v2': 4.3, 'v3': 3.9}

def gen_and_judge_mock(model, style, case):
    prof = MODEL_PROFILE[model]
    base = min(5.0, STYLE_BASE[style] + prof['depth_bonus'])
    def s(mu): return int(min(5, max(1, round(random.gauss(mu, 0.5)))))
    d = {'openness': s(base if style!='v1' else base-0.3),
         'groundedness': s(base+0.2), 'specificity': s(base-0.2),
         'provocativeness': s(base if style=='v3' else base-0.1)}
    # 환각: v1 + mini에서 가장 잦음, gpt-4o는 절반 수준
    fab = 1 if (style=='v1' and random.random() < prof['fab_base']) else 0
    if fab: d['groundedness'] = min(d['groundedness'], 2)
    depth = round(sum(d[k]*w for k,w in W.items()), 3)
    return {'depth_score': depth, 'fabrication': fab,
            'cost_usd': max(0.0, random.gauss(prof['cost'], prof['cost']*0.15)),
            'latency_ms': max(200, int(random.gauss(prof['lat'], 350))), **d}

def gen_and_judge_api(model, style, case):
    # 라이브: 생성(모델별) → 판정. 백엔드 대신 OpenAI 직접 호출로 '모델'을 바꿔 비교.
    from openai import OpenAI
    import yaml
    client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    prompts = yaml.safe_load(open('../configs/prompts.yaml', encoding='utf-8'))
    sysp = prompts['system'][style].format(book_title=case['book_query'], book_author='',
                                           book_summary='(요약 생략)')
    t0=time.time()
    r = client.chat.completions.create(model=model, temperature=0.7, max_tokens=400,
        messages=[{'role':'system','content':sysp},{'role':'user','content':case['user_msg']}])
    lat=int((time.time()-t0)*1000); amsg=r.choices[0].message.content or ''
    rub = ('4축(openness/groundedness/specificity/provocativeness) 1~5 + fabrication(0/1). JSON만.\n'
           + '[사용자]' + case['user_msg'] + '\n[응답]' + amsg)
    jr = client.chat.completions.create(model=JUDGE_MODEL, temperature=0.0, max_tokens=200,
        messages=[{'role':'user','content':rub}], response_format={'type':'json_object'})
    d=json.loads(jr.choices[0].message.content); d.setdefault('fabrication',0)
    d['depth_score']=round(sum(int(d.get(k,3))*w for k,w in W.items()),3)
    d['latency_ms']=lat; d['cost_usd']=0.0  # (비용은 usage로 환산 가능, 데모 생략)
    return d

rows=[]
for model in MODELS:
    for style in STYLES:
        for case in GOLDEN:
            j = gen_and_judge_api(model,style,case) if USE_API else gen_and_judge_mock(model,style,case)
            j.update({'model':model,'style':style,'book':case['book_query']})
            rows.append(j)
            if USE_API: time.sleep(0.2)
df = pd.DataFrame(rows)
print('총 실험 응답:', len(df), '= ', len(MODELS),'모델 ×',len(STYLES),'스타일 ×',len(GOLDEN),'케이스')
df.head()

## 2. 집계 — 모델 × 스타일 성능 비교 (#6)

In [ ]:
agg = (df.groupby(['model','style'])
         .agg(depth=('depth_score','mean'), fab_rate=('fabrication','mean'),
              cost=('cost_usd','mean'), latency=('latency_ms','mean'))
         .round(4).reset_index())
pivot = agg.pivot(index='style', columns='model', values='depth')
print('Question Depth (모델 × 스타일):'); print(pivot.round(2))
print('\n모델별 종합:')
model_overall = (df.groupby('model')
    .agg(depth=('depth_score','mean'), fab_rate=('fabrication','mean'),
         cost=('cost_usd','mean'), latency=('latency_ms','mean')).round(4))
print(model_overall)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2,figsize=(12,4))
pivot.plot(kind='bar', ax=ax[0]); ax[0].axhline(4.0,color='gray',ls='--')
ax[0].set_title('Question Depth: model x style'); ax[0].set_ylim(0,5); ax[0].legend(title='model')
model_overall[['cost']].plot(kind='bar', ax=ax[1], legend=False, color=['#1F4E79','#C0504D'])
ax[1].set_title('Avg cost per response (USD)')
plt.tight_layout(); plt.savefig('model_comparison.png', dpi=120); plt.show()

## 3. MLflow로 실험 추적 (#5)

모델별로 하나의 run을 기록한다 — params(model, judge, styles, golden 규모, mode)와
metrics(스타일별 depth, 종합 depth, 환각률, 평균 비용/지연). MLflow UI(`mlflow ui`)에서
실험 간 비교·정렬·재현이 가능하다. 미설치 시 `experiments/runs.csv`로 폴백한다.

In [ ]:
os.makedirs('../experiments', exist_ok=True)
RUNS_CSV = '../experiments/runs.csv'

def log_fallback_csv(record):
    df1 = pd.DataFrame([record])
    df1.to_csv(RUNS_CSV, mode='a', header=not os.path.exists(RUNS_CSV), index=False)

try:
    os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'   # 로컬 파일 스토어 허용(데모)
    import mlflow
    mlflow.set_tracking_uri('file:../mlruns')
    mlflow.set_experiment('reading-coach-model-comparison')
    USE_MLFLOW = True
except Exception as e:
    USE_MLFLOW = False
    print('MLflow 미사용 → CSV 폴백:', e)

for model in MODELS:
    sub = df[df['model']==model]
    by_style = sub.groupby('style')['depth_score'].mean().round(3).to_dict()
    record = {
        'model': model, 'judge_model': JUDGE_MODEL, 'mode': 'live' if USE_API else 'offline_mock',
        'n_golden': len(GOLDEN), 'styles': '|'.join(STYLES),
        'depth_overall': round(float(sub['depth_score'].mean()),3),
        'depth_v1': by_style.get('v1'), 'depth_v2': by_style.get('v2'), 'depth_v3': by_style.get('v3'),
        'fabrication_rate': round(float(sub['fabrication'].mean()),4),
        'avg_cost_usd': round(float(sub['cost_usd'].mean()),6),
        'avg_latency_ms': round(float(sub['latency_ms'].mean()),1),
    }
    if USE_MLFLOW:
        with mlflow.start_run(run_name=model):
            mlflow.log_params({k:record[k] for k in ['model','judge_model','mode','n_golden','styles']})
            mlflow.log_metrics({k:v for k,v in record.items()
                                if isinstance(v,(int,float)) and v is not None})
    else:
        log_fallback_csv(record)
    print('logged run:', model, '| depth_overall=', record['depth_overall'])

print('\nMLflow 추적 위치:', os.path.abspath('../mlruns') if USE_MLFLOW else RUNS_CSV)
print('UI: mlflow ui --backend-store-uri ./mlruns  (http://localhost:5000)')

## 4. 채택 결정 + 산출물 저장

비용 대비 품질로 채택안을 결정한다. (gpt-4o가 depth는 높아도 비용이 ~8배면
MVP는 gpt-4o-mini + v2를 채택하고, gpt-4o는 고난도 케이스 옵션으로 둘 수 있다.)

In [ ]:
best = agg.sort_values('depth', ascending=False).iloc[0]
# 비용효율: 동일 depth대면 저비용 우선. 간단한 점수 = depth - 비용패널티
agg['value'] = agg['depth'] - agg['cost']*120   # 비용 민감 가중(MVP는 비용효율 중시)
reco = agg.sort_values('value', ascending=False).iloc[0]

decision = {
    'best_quality': {'model': best['model'], 'style': best['style'], 'depth': float(best['depth'])},
    'best_value':   {'model': reco['model'], 'style': reco['style'],
                     'depth': float(reco['depth']), 'cost': float(reco['cost'])},
    'mode': 'live' if USE_API else 'offline_mock',
    'models_compared': MODELS,
}
json.dump(decision, open('model_comparison.json','w',encoding='utf-8'), ensure_ascii=False, indent=2)
print(json.dumps(decision, ensure_ascii=False, indent=2))

## 5. 요약

- **#5 실험 관리:** 모델×프롬프트 조합을 체계적으로 탐색하고 MLflow로 추적 → 재현·비교 가능.
- **#6 모델 비교:** 동일 골든셋에서 gpt-4o-mini vs gpt-4o의 Depth/환각/비용/지연을 정량 비교.
- **결정:** 품질 최고안과 비용효율 최고안을 분리 산출(`model_comparison.json`).
- 다음: `03_evaluation_calibration.ipynb`(채택안의 사람 보정), `04_xai.ipynb`(무엇이 좋은 질문을 만드는가).